# Explainable AI (XAI)
- **Occlusion Sensitivity Heatmaps**:
    - Occlusion sensitivity heatmaps (`occlusion_patch_size = 32`, `occlusion_stride = 16`) localize influential texture regions with manageable compute.
    - Occlusion tests the model’s response when local patches are masked, producing an importance map over the image.
    - It is useful for texture-driven models because it reveals whether the classifier depends on local texture regions instead of only global statistics.
    - Advantages: highlights regions that affect the decision, helps detect whether the model is using meaningful defect texture, and provides a sanity-check for trust and debugging.
    - Limitations: patch-based heatmaps are coarse, can be noisy for texture models, and may reflect correlated texture patterns rather than exact physical defect shapes.
    - Practical caveat: the attention may appear patchy and texture-driven, not aligned to consistent defect shapes, so use heatmaps as diagnostic guidance rather than precise localization.

#TODO interpret model using generated heatmaps . doc as approach + parameters

In [ ]:
from pathlib import Path

from attr import dataclass
from matplotlib import pyplot as plt
from matplotlib.image import imread
import numpy as np
from skimage.transform import resize
from sklearn.pipeline import Pipeline
from steelblast_classic_features import (FeatureExtractionConfig, extract_features_from_image, read_grayscale_image)
from steelblast_classic_features import CLASS_NAMES, LABELS

@dataclass(frozen=True)
class HeatmapConfig:
    heatmap_dir: Path = Path("steelblast_svm_glcm_dwt_focus_heatmaps")
    image_size: int = 256
    occlusion_patch_size: int = 32
    occlusion_stride: int = 16
    max_per_case: int | None = 3
    occlusion_fill_mode: str = "median"  # median or zero
    clear_output_dir: bool = True


heatmap_config = HeatmapConfig()


def read_display_image(image_path: Path, image_size: int) -> np.ndarray:
    image = imread(image_path)

    if image.ndim == 2:
        image = np.repeat(image[..., np.newaxis], 3, axis=2)
    else:
        image = image[..., :3]

    image = resize(
        image,
        (image_size, image_size),
        anti_aliasing=True,
        preserve_range=True,
    )

    image = image.astype(np.float32)
    if image.max() > 1.0:
        image /= 255.0

    return np.clip(image, 0.0, 1.0)


def save_focus_pair(
    display_image: np.ndarray,
    heatmap: np.ndarray,
    title: str,
    output_path: Path,
) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), constrained_layout=True)
    axes[0].imshow(display_image)
    axes[0].set_title("Example image")
    axes[0].set_axis_off()

    axes[1].imshow(display_image)
    vmax = float(heatmap.max()) if heatmap.size and heatmap.max() > 0 else 1.0
    overlay = axes[1].imshow(heatmap, cmap="jet", alpha=0.62, vmin=0, vmax=vmax)
    axes[1].set_title("Activation heatmap")
    axes[1].set_axis_off()
    fig.colorbar(overlay, ax=axes[1], label="Decision-score drop")
    fig.suptitle(title, fontsize=16)
    fig.savefig(output_path, dpi=180)
    plt.close(fig)


def compute_occlusion_focus_heatmap(
    image: np.ndarray,
    predicted_label: int,
    score_fn,
    patch_size: int,
    stride: int,
    fill_value: float,
) -> np.ndarray:
    """Generic occlusion engine decoupled from model internals."""
    base_confidence = score_fn(image, predicted_label)
    heatmap = np.zeros_like(image, dtype=np.float32)
    counts = np.zeros_like(image, dtype=np.float32)
    height, width = image.shape

    for top in range(0, height, stride):
        bottom = min(top + patch_size, height)
        top = max(0, bottom - patch_size)

        for left in range(0, width, stride):
            right = min(left + patch_size, width)
            left = max(0, right - patch_size)

            occluded = image.copy()
            occluded[top:bottom, left:right] = fill_value
            occluded_confidence = score_fn(occluded, predicted_label)
            importance = max(0.0, base_confidence - occluded_confidence)

            heatmap[top:bottom, left:right] += importance
            counts[top:bottom, left:right] += 1

    return np.divide(heatmap, counts, out=np.zeros_like(heatmap), where=counts > 0)


def resolve_feature_adapter():
    """
    Build local image->feature adapter without requiring global `feature_config`.
    Uses default FeatureExtractionConfig if one is not present in notebook state.
    """
    %store -r feature_config
    if feature_config is None:
        cfg = FeatureExtractionConfig(
            image_size=256, # image size to resize to (256x256)
            illumination_normalization="clahe", # illumination normalization method
            glcm_levels=32, # Number of gray levels for GLCM quantization
            glcm_properties=("contrast",
                            "dissimilarity",
                                "homogeneity",
                                "energy",
                                    "correlation", "ASM"), # GLCM properties to compute
            dwt_wavelet="db2", # Wavelet type for DWT
            dwt_level=3 # Decomposition level for DWTs
        )
        print("feature_config not found in kernel state; using default FeatureExtractionConfig()")
    else:
        cfg = feature_config

    def image_to_features_fn(image: np.ndarray) -> np.ndarray:
        return extract_features_from_image(image, cfg)

    def grayscale_loader(image_path: Path, image_size: int) -> np.ndarray:
        return read_grayscale_image(image_path, image_size, cfg)

    return image_to_features_fn, grayscale_loader


def make_svm_score_fn(fitted_model: Pipeline, image_to_features_fn):
    """Return image->label confidence adapter for SVM using a generic feature adapter."""

    def _score_fn(image: np.ndarray, label: int) -> float:
        features = image_to_features_fn(image).reshape(1, -1)
        scores = fitted_model.decision_function(features)

        if np.ndim(scores) == 1:
            positive_class = int(fitted_model.classes_[1])
            score = float(scores[0])
            return score if label == positive_class else -score

        class_index = int(np.flatnonzero(fitted_model.classes_ == label)[0])
        return float(scores[0, class_index])

    return _score_fn


def svm_predicted_label_confidences(
    fitted_model: Pipeline,
    features: np.ndarray,
    predictions: np.ndarray,
) -> np.ndarray:
    """Confidence per sample for its predicted class (used for ranking exports)."""
    scores = fitted_model.decision_function(features)
    if np.ndim(scores) == 1:
        positive_class = int(fitted_model.classes_[1])
        signed_scores = np.where(predictions == positive_class, scores, -scores)
        return signed_scores.astype(np.float64)

    return np.asarray(
        [scores[index, np.flatnonzero(fitted_model.classes_ == label)[0]] for index, label in enumerate(predictions)],
        dtype=np.float64,
    )


import shutil


def reset_directory(output_dir: Path, clear_output_dir: bool) -> None:
    if clear_output_dir and output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)


def save_classification_case_heatmaps(
    test_paths: list[Path],
    y_test: np.ndarray,
    y_pred: np.ndarray,
    X_test: np.ndarray,
    fitted_model: Pipeline,
    heatmap_config: HeatmapConfig,
) -> dict[str, list[dict[str, object]]]:
    """Generate and save occlusion heatmaps for TP, FP, FN, TN."""
    reset_directory(heatmap_config.heatmap_dir, heatmap_config.clear_output_dir)

    image_to_features_fn, grayscale_loader = resolve_feature_adapter()
    heatmap_results: dict[str, list[dict[str, object]]] = {}
    confidences = svm_predicted_label_confidences(fitted_model, X_test, y_pred)
    score_fn = make_svm_score_fn(fitted_model, image_to_features_fn)

    case_definitions = {
        "true_positive": (1, 1),
        "false_positive": (0, 1),
        "false_negative": (1, 0),
        "true_negative": (0, 0),
    }

    for case_name, (actual_label, predicted_label) in case_definitions.items():
        case_indices = [
            i
            for i, (actual, predicted) in enumerate(zip(y_test, y_pred))
            if actual == actual_label and predicted == predicted_label
        ]

        case_indices = sorted(
            case_indices,
            key=lambda i: confidences[i],
            reverse=True,
        )

        if heatmap_config.max_per_case is not None:
            case_indices = case_indices[:heatmap_config.max_per_case]

        case_dir = heatmap_config.heatmap_dir / case_name
        case_dir.mkdir(parents=True, exist_ok=True)

        heatmap_results[case_name] = []
        print(f"\nProcessing {case_name}: {len(case_indices)} images")

        for rank, idx in enumerate(case_indices):
            try:
                image_path = test_paths[idx]
                grayscale_image = grayscale_loader(image_path, heatmap_config.image_size)
                display_image = read_display_image(image_path, heatmap_config.image_size)

                if heatmap_config.occlusion_fill_mode == "median":
                    fill_value = float(np.median(grayscale_image))
                elif heatmap_config.occlusion_fill_mode == "zero":
                    fill_value = 0.0
                else:
                    raise ValueError("occlusion_fill_mode must be 'median' or 'zero'")

                heatmap = compute_occlusion_focus_heatmap(
                    grayscale_image,
                    int(y_pred[idx]),
                    score_fn,
                    heatmap_config.occlusion_patch_size,
                    heatmap_config.occlusion_stride,
                    fill_value,
                )

                output_path = case_dir / f"{rank:03d}_{image_path.stem}_heatmap.png"
                title = (
                    f"{case_name.replace('_', ' ').title()} | "
                    f"actual {CLASS_NAMES[int(y_test[idx])]}, "
                    f"predicted {CLASS_NAMES[int(y_pred[idx])]} | "
                    f"conf {confidences[idx]:.3f}"
                )

                save_focus_pair(display_image, heatmap, title, output_path)

                result = {
                    "index": int(idx),
                    "rank": rank,
                    "actual_label": CLASS_NAMES[int(y_test[idx])],
                    "predicted_label": CLASS_NAMES[int(y_pred[idx])],
                    "source_image": str(image_path),
                    "heatmap": str(output_path),
                    "confidence": float(confidences[idx]),
                    "max_activation": float(np.max(heatmap)),
                }
                heatmap_results[case_name].append(result)
            except Exception as e:
                print(f"Failed on index {idx}: {e}")

        print(f"Done {case_name}: saved {len(heatmap_results[case_name])}")

    return heatmap_results



%store -r model_svm
fitted_model_svm = model_svm.best_estimator_ if hasattr(model_svm, "best_estimator_") else model_svm
%store -r X_test_svm
%store -r y_pred_svm
%store -r test_paths_svm
%store -r y_test_svm
heatmap_paths = save_classification_case_heatmaps(
    test_paths_svm, #paths
    y_test_svm, #labels
    y_pred_svm, #predicted labels
    X_test_svm, #feature matrix. dependent on the model used. For SVM, it is the feature matrix of test images
    fitted_model_svm,
    heatmap_config,
)